In [1]:
import glob
import sys
from tqdm import tqdm
import os

root_dir = "/projects/bbhh/suyufeng/enzyme_specificity"

sys.path.append(f"{root_dir}/src")

# Extract pocket

In [7]:
from Bio.PDB import *
from rdkit import Chem
import glob
from tqdm import tqdm
from ray.util.multiprocessing import Pool
import os
import ray

@ray.remote
def single_extract_pocket(names):
    bad_mol = 0
    for name in names:
        id = int(name.split("/")[-1].split(".")[0])

        pocket_out_path = f"/scratch/bbto/suyufeng/data/pocket/{id}.pdb"
        ligand_out_path = f"/scratch/bbto/suyufeng/data/raw_ligand/{id}.sdf"
        lines = []
        ed = 0
        for index, line in enumerate(open(name, "r")):
            lines.append(line)
            if "COMPND" in line:
                ed = index

        protein_lines = lines[:ed]
        ligand_lines = lines[ed+1:]

        ftmp_out = open(f"/scratch/bbto/suyufeng/data/raw_ligand/tmp_{id}.pdb", "w")
        for line in ligand_lines:
            ftmp_out.write(line)
        ftmp_out.close()

        mol = Chem.MolFromPDBFile(f"/scratch/bbto/suyufeng/data/raw_ligand/tmp_{id}.pdb", flavor=1, sanitize=False)
        os.system(f"rm -r /scratch/bbto/suyufeng/data/raw_ligand/tmp_{id}.pdb")
        if mol is None:
            print(f"!!!!!{id}!!!!!")
            bad_mol += 1
            continue

        ligand_coords = []
        # mol = Chem.RemoveHs(mol, sanitize=False)
        for i, atom in enumerate(mol.GetAtoms()):
            positions = mol.GetConformer().GetAtomPosition(i)
            ligand_coords.append((positions.x, positions.y, positions.z))
        
        try:
            writer = Chem.SDWriter(ligand_out_path)
            writer.write(mol, confId=0)
        except:
            bad_mol += 1
            continue

        fin = open(pocket_out_path, "w")

        for line in protein_lines:
            if "ATOM" in line:
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    for ligand_coord in ligand_coords:
                        if 'H' not in line[12:16].strip() and distance(x, y, z, ligand_coord[0], ligand_coord[1], ligand_coord[2]) < 10:
                            fin.write(line)
                            break
                except:
                    continue
                
            elif "HETATM" in line or "ENDMDL" in line:
                fin.write(line)
    return bad_mol

def distance(x1, y1, z1, x2, y2, z2):
    return (1. * (x2 - x1) ** 2 + 1. * (y2 - y1) ** 2 + 1. * (z2 - z1) ** 2) ** 0.5


os.makedirs("/scratch/bbto/suyufeng/data", exist_ok=True)
os.makedirs("/scratch/bbto/suyufeng/data/raw_ligand", exist_ok=True)
os.makedirs("/scratch/bbto/suyufeng/data/pocket", exist_ok=True)

mol = None

parameters = []
for name in tqdm(glob.glob(f"/scratch/bbto/tjdean2/General_dataset/*.pdb")):
    parameters.append(name)

results = [parameters[i*1000: (i+1)*1000] for i in range(len(parameters) // 1000)]
bad_molecular = 0


results = [single_extract_pocket.remote(parameters[i*100: (i+1)*100]) for i in range(len(parameters) // 100)]
bad_molecular = 0

for result in tqdm(ray.get(results)):
    bad_molecular += result
    
# print(single_extract_pocket(parameters))
print(bad_molecular)

100%|██████████| 534213/534213 [00:00<00:00, 3772109.20it/s]


9313
-73.414 -39.919 -5.745 168.62899768130035
-73.324 -40.714 -6.344 168.91626334074527
-74.345 -39.566 -5.841 169.34219913831282
-73.299 -40.23 -4.802 168.58465881568227
-72.411 -38.877 -6.088 167.31871831029542
-71.545 -39.32 -5.855 166.69159956338532
-72.474 -37.615 -5.218 166.808058924022
-71.448 -37.279 -4.65 165.69638584471295
-72.401 -38.556 -7.585 167.30041519673526
-73.334 -38.596 -7.941 168.19813717755616
-72.028 -37.639 -7.728 166.6059995258274
-71.529 -39.583 -8.315 166.98547264358058
-70.604 -39.413 -7.976 166.04579167807896
-71.85 -40.469 -7.98 167.61063903583207
-71.615 -39.475 -10.114 167.18944319842686
-70.877 -37.846 -10.425 165.89233663132242
-70.856 -37.643 -11.404 165.89522164908786
-69.938 -37.807 -10.083 164.98713899877166
-71.399 -37.124 -9.972 166.03527840793353
-73.619 -36.932 -5.032 167.5756057067973
-74.457 -37.276 -5.455 168.5088671049687
-73.687 -35.685 -4.221 167.1005449841502
-73.015 -35.097 -4.671 166.28570049766753
-73.235 -35.827 -2.752 166.650618531

0

In [5]:
from rdkit.Chem import rdFMCS
from rdkit.Chem import Draw
from rdkit.Chem import AllChem,rdDepictor
from rdkit import Chem
import pandas as pd
import glob
from rdkit import RDLogger
import ray

root_dir = "/projects/bbto/suyufeng/enzyme_specificity"

def AssignBondOrdersFromTemplate(refmol, mol):
    """ assigns bond orders to a molecule based on the
        bond orders in a template molecule
    Revised from AllChem.AssignBondOrderFromTemplate(refmol, mol)
    """
    AllChem.AssignBondOrdersFromTemplate
    refmol2 = Chem.rdchem.Mol(refmol)
    mol2 = Chem.rdchem.Mol(mol)
    # do the molecules match already?
    matching = mol2.GetSubstructMatch(refmol2)
    if not matching:  # no, they don't match
        # check if bonds of mol are SINGLE
        for b in mol2.GetBonds():
            if b.GetBondType() != Chem.BondType.SINGLE:
                b.SetBondType(Chem.BondType.SINGLE)
                b.SetIsAromatic(False)
        # set the bonds of mol to SINGLE
        for b in refmol2.GetBonds():
            b.SetBondType(Chem.BondType.SINGLE)
            b.SetIsAromatic(False)
        # set atom charges to zero;
        for a in refmol2.GetAtoms():
            a.SetFormalCharge(0)
        for a in mol2.GetAtoms():
            a.SetFormalCharge(0)

        matching = mol2.GetSubstructMatches(refmol2, uniquify=False)
        # do the molecules match now?
        if matching:
            if len(matching) > 1:
                #logger.warning("More than one matching pattern found - picking one")
                pass
            matchings=matching[:]
            for matching in matchings:
                #matching = matching[0] ## use each matching
                # apply matching: set bond properties
                for b in refmol.GetBonds():
                    atom1 = matching[b.GetBeginAtomIdx()]
                    atom2 = matching[b.GetEndAtomIdx()]
                    b2 = mol2.GetBondBetweenAtoms(atom1, atom2)
                    b2.SetBondType(b.GetBondType())
                    b2.SetIsAromatic(b.GetIsAromatic())
                # apply matching: set atom properties
                for a in refmol.GetAtoms():
                    a2 = mol2.GetAtomWithIdx(matching[a.GetIdx()])
                    a2.SetHybridization(a.GetHybridization())
                    a2.SetIsAromatic(a.GetIsAromatic())
                    a2.SetNumExplicitHs(a.GetNumExplicitHs())
                    a2.SetFormalCharge(a.GetFormalCharge())
                try:
                    Chem.SanitizeMol(mol2)
                    if hasattr(mol2, '__sssAtoms'):
                        mol2.__sssAtoms = None  # we don't want all bonds highlighted
                    break
                except ValueError:
                    pass
                    # print("More than one matching pattern, Fail at this matching. Try next.")
        else:
            raise ValueError("No matching found")
    return mol2

def alignment_number_system(sdf, smile_mol):
    
    template = smile_mol
    query = sdf

    mcs = rdFMCS.FindMCS([template, query], timeout=120)
    patt = Chem.MolFromSmarts(mcs.smartsString)

    query_match = query.GetSubstructMatch(patt)
    template_match = template.GetSubstructMatch(patt)

    result = [-1] * query.GetNumAtoms()

    for query_atom_id, template_atom_id in zip(query_match, template_match):
        result[query_atom_id] = template_atom_id

    # Check if there is any atom not matched
    for atom in query.GetAtoms():
        assert atom.GetAtomicNum() == 1 or result[atom.GetIdx()] != -1

    return result

def assign_idx(mol, idxs):
    for atom, idx in zip(mol.GetAtoms(), idxs):
        atom.SetAtomMapNum(idx)
    return mol

def mol_get_atomic_number(mol, atom_map=False):
    result = [0] * mol.GetNumAtoms()
    for atom in mol.GetAtoms():
        if atom.GetAtomMapNum() != -1:
            if atom_map:
                result[atom.GetAtomMapNum()] = atom.GetAtomicNum()
            else:
                result[atom.GetIdx()] = atom.GetAtomicNum()
    return result

def check(mol, mol2):
    for atom in mol.GetAtoms():
        if atom.GetAtomMapNum() != -1:
            id = atom.GetAtomMapNum()
            
            atom2 = mol2.GetAtomWithIdx(id)
            if atom.GetAtomicNum() != atom2.GetAtomicNum():
                return False
    return True

def view_difference(mol1, mol2):
    mcs = rdFMCS.FindMCS([mol1,mol2])
    mcs_mol = Chem.MolFromSmarts(mcs.smartsString)
    match1 = mol1.GetSubstructMatch(mcs_mol)
    target_atm1 = []
    for atom in mol1.GetAtoms():
        if atom.GetIdx() not in match1:
            target_atm1.append(atom.GetIdx())
    match2 = mol2.GetSubstructMatch(mcs_mol)
    target_atm2 = []
    for atom in mol2.GetAtoms():
        if atom.GetIdx() not in match2:
            target_atm2.append(atom.GetIdx())
    return Draw.MolsToGridImage([mol1, mol2],highlightAtomLists=[target_atm1, target_atm2])

def single_match(item):
    id, smile = item
    sdf_path = f"/scratch/bbto/suyufeng/data/raw_ligand/{id}.sdf"
    RDLogger.DisableLog('rdApp.*')

    try:
        mol = next(iter(Chem.SDMolSupplier(sdf_path, sanitize=True)))
        mol = Chem.RemoveHs(mol)
    except Exception as e:
        try:
            mol = next(iter(Chem.SDMolSupplier(sdf_path, sanitize=False)))
            mol = Chem.RemoveHs(mol, sanitize=False)
        except Exception as e:
            return 1 
    # mol = Chem.MolFromSmiles(Chem.MolToSmiles(mol))
    try:
        smile_mol = Chem.MolFromSmiles(smile)
    except Exception as e:
        return 1
    # print(mol_get_atomic_number(smile_mol))
    # print(len(smile_mol.GetAtoms()))
    try:
        aligned_idx = alignment_number_system(mol, smile_mol)
    except:
        
        try:
            mol = AssignBondOrdersFromTemplate(smile_mol, mol)
        except:
            return 1
        
        try:
            aligned_idx = alignment_number_system(mol, smile_mol)
        except:
            return 1
    mol = assign_idx(mol, aligned_idx)
    # print(mol_get_atomic_number(mol, atom_map=True))
    if not check(mol, smile_mol):
        return 1

    w = Chem.SDWriter(f"/scratch/bbto/suyufeng/data/ligand/{id}.sdf")
    try:
        w.write(mol)
        w.close()
    except:
        w.close()
        return 1
    
    return 0

@ray.remote(num_cpus=1)
def batched_match(items):
    ans = 0
    for item in items:
        id, smile = item
        try:
            result = single_match(item)
            ans += result
        except:
            ans += 1

        if not result:
            os.system(f"rm -r /scratch/bbto/suyufeng/data/raw_ligand/{id}.sdf")

    return ans

from tqdm import tqdm

import os
os.makedirs(f"/scratch/bbto/suyufeng/data/ligand", exist_ok=True)

df = pd.read_csv(f"{root_dir}/data/brenda/final_data/data.csv")
sub_index_dict = {index: substrate_id for index, substrate_id in zip(df["structure_index"].values, df["reaction"].values)}


sub_df = pd.read_csv(f"{root_dir}/data/brenda/reaction.csv")
substrates = sub_df["substrates"].values

bad_molecular = 0

parameters = []
for index, sdf_path in tqdm(enumerate(glob.glob(f"/scratch/bbto/suyufeng/data/raw_ligand/*.sdf"))):
    
    id = os.path.basename(sdf_path).split(".")[0]
    
    if int(id) not in sub_index_dict:
        continue

    substrate_id = int(sub_index_dict[int(id)])
    smile = substrates[substrate_id]

    parameters.append((int(id), smile))

results = [batched_match.remote(parameters[i*100: (i+1)*100]) for i in range(len(parameters) // 100)]
bad_molecular = 0

for result in tqdm(ray.get(results)):
    bad_molecular += result
    
    # break
print(bad_molecular)

0
